# Module 3.5: Retention & Decay

As an agent accumulates memories over months, a problem emerges: **retrieval
quality degrades**. The more memories stored, the more noise competes with signal.
Eventually, asking "What hotel does Sarah prefer?" returns a mix of genuine
preferences and one-off mentions from months ago.

Simple cache heuristics (LRU, LFU) don't work — they can't distinguish a rarely-used
but critical allergy fact from frequently-retrieved but low-value noise.

> **The question**: How do we keep memory useful as it grows?

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import sys, os, json, random, math
import sniffio
from datetime import datetime, timezone, timedelta

sys.path.insert(0, "..")
sniffio.current_async_library_cvar.set("asyncio")

from agent_framework._types import Message
from lifecycle_utils import (
    MemoryItem, MemoryState, RetentionScorer
)
from shared.travel_agent import create_client

client, credential = create_client("../.env")
print("Setup complete")

## The Problem: Unbounded Memory Kills Retrieval

What happens when we never evict anything? Let's simulate 6 months of memory
accumulation with realistic noise levels and measure retrieval precision.

In [ ]:
import random
from datetime import datetime, timezone, timedelta

# Sarah's REAL preferences (signal)
REAL_PREFERENCES = [
    "Prefers Marriott hotels",
    "Vegetarian diet",
    "Window seat on long flights",
    "Home airport is SFO",
    "Budget max $250/night",
]

# 6 months of noise that accumulated alongside real preferences
ACCUMULATED_NOISE = [
    "Asked about weather in Tokyo",
    "Colleague Mike mentioned Hilton once",
    "Saw an ad for Spirit Airlines",
    "Discussed rental car prices briefly",
    "Random observation about JFK airport",
    "Asked about train passes in Europe",
    "Weather was cold in London",
    "Mentioned duty free shopping",
    "Colleague Alex recommended travel insurance",
    "Talked about airport lounges once",
    "Asked about currency exchange rates",
    "Mentioned a Netflix documentary about airlines",
    "Discussed commute times to airport",
    "Weather was rainy in Seattle",
    "Asked about pet travel policies",
]

# Simulate: an unbounded store with everything
unbounded_store = REAL_PREFERENCES + ACCUMULATED_NOISE
random.shuffle(unbounded_store)

# Retrieve "top 5" for query "hotel preferences" (simulated keyword match)
hotel_keywords = ["hotel", "marriott", "hilton", "night", "budget"]
retrieved = [m for m in unbounded_store
             if any(k in m.lower() for k in hotel_keywords)]

print(f"=== Unbounded Store: {len(unbounded_store)} memories ===\n")
print(f"Query: 'What are Sarah's hotel preferences?'")
print(f"Retrieved {len(retrieved)} results:\n")
for m in retrieved:
    is_signal = m in REAL_PREFERENCES
    icon = "✅" if is_signal else "❌"
    print(f"  {icon} {m}")

signal_count = sum(1 for m in retrieved if m in REAL_PREFERENCES)
print(f"\nPrecision: {signal_count}/{len(retrieved)} = "
      f"{signal_count/max(len(retrieved),1):.0%}")
print(f"\n→ Noise ratio in store: {len(ACCUMULATED_NOISE)}/{len(unbounded_store)} = "
      f"{len(ACCUMULATED_NOISE)/len(unbounded_store):.0%}")
print(f"→ As more noise accumulates, precision drops further")
print(f"→ After 1 year: ~90% noise → retrieval becomes nearly useless")

## What Went Wrong

The unbounded store has no way to distinguish valuable memories from noise:

| Problem | Impact |
|---------|--------|
| **No recency weighting** | A 6-month-old offhand mention ranks equally with a confirmed preference |
| **No usage tracking** | Memories retrieved 0 times sit alongside frequently-used ones |
| **No success signal** | Memories that led to bad recommendations are kept forever |
| **Linear growth** | Storage cost grows forever; context windows overflow |

The core issue: **not all memories are equally valuable, but the store treats them as if they are.**

> *"Selective Memory Retention for Long-Horizon LLM Agents"* (arXiv:2606.29178) —
> Under 75% noise injection, unbounded memory degrades Precision@5 from
> 20.2% to 12.4%, while bounded retention maintains 16.9% → 16.6%.

## The Solution: Multi-Dimensional Scoring with Bounded Capacity

Instead of treating all memories equally, we **score** them on multiple dimensions
and evict the lowest-scored when at capacity. This is NOT a simple cache (LRU/LFU)
— it considers semantic value, success correlation, and trust state.

| Dimension | Weight | Meaning |
|-----------|--------|----------|
| Recency | 0.20 | Newer memories score higher (exponential decay) |
| Access frequency | 0.15 | Frequently retrieved memories score higher |
| Success correlation | 0.25 | Memories that led to good outcomes score highest |
| Redundancy penalty | 0.15 | Similar-to-others memories score lower |
| Specificity | 0.10 | Detailed, specific memories score higher |
| Utility (state-based) | 0.15 | Trusted > Provisional > Candidate |

In [ ]:
scorer = RetentionScorer(
    capacity=20,            # Small capacity for demonstration
    half_life_days=30.0,    # Memories decay with 30-day half-life
)

print(f"RetentionScorer configuration:")
print(f"  Capacity: {scorer.capacity}")
print(f"  Half-life: {scorer.half_life_days} days")
print(f"  Weights:")
for dim, weight in scorer.weights.items():
    print(f"    {dim:<20} = {weight:.2f}")

In [ ]:
# Create sample memories with varying characteristics
now = datetime.now(timezone.utc)

sample_memories = [
    # High-value: recent, accessed often, trusted, successful
    MemoryItem(
        user_id="E001", content="Prefers Marriott hotels in NYC",
        category="preference", state=MemoryState.TRUSTED,
        first_seen=now - timedelta(days=10), access_count=8,
        last_accessed=now - timedelta(days=1),
        success_correlation=0.9, confirmation_count=4,
    ),
    # Medium-value: moderate age, some access
    MemoryItem(
        user_id="E001", content="Usually takes morning flights",
        category="preference", state=MemoryState.PROVISIONAL,
        first_seen=now - timedelta(days=45), access_count=3,
        last_accessed=now - timedelta(days=15),
        success_correlation=0.4, confirmation_count=1,
    ),
    # Low-value: old, never accessed, candidate, no success
    MemoryItem(
        user_id="E001", content="Mentioned liking jazz music once",
        category="preference", state=MemoryState.CANDIDATE,
        first_seen=now - timedelta(days=120), access_count=0,
        last_accessed=None,
        success_correlation=0.0, confirmation_count=0,
    ),
    # Stale-but-specific: old, but very detailed and trusted
    MemoryItem(
        user_id="E001",
        content="Has severe peanut allergy — requires airline notification 48h before flight, carries EpiPen, needs allergen-free hotel room",
        category="fact", state=MemoryState.TRUSTED,
        first_seen=now - timedelta(days=200), access_count=5,
        last_accessed=now - timedelta(days=60),
        success_correlation=0.7, confirmation_count=3,
    ),
]

print("=== Retention Scores ===")
print()
for m in sample_memories:
    score = scorer.score(m)
    age_days = (now - m.first_seen).days
    print(f"  Score: {score:.3f} | {m.content[:50]}...")
    print(f"          Age: {age_days}d | Accessed: {m.access_count}x | "
          f"State: {m.state.value} | Success: {m.success_correlation}")
    print()

## Building the Bounded Store

When the store reaches capacity, the scorer ranks all memories and evicts
the lowest-scored ones. This keeps memory quality high over time.

In [ ]:
class BoundedMemoryStore:
    """Memory store with capacity limits and score-based eviction."""

    def __init__(self, capacity: int = 20, scorer: RetentionScorer = None):
        self.capacity = capacity
        self.scorer = scorer or RetentionScorer(capacity=capacity)
        self.memories: list[MemoryItem] = []
        self.eviction_log: list[dict] = []

    def add(self, memory: MemoryItem) -> list[MemoryItem]:
        """Add a memory. Returns list of evicted memories if at capacity."""
        self.memories.append(memory)
        evicted = []
        if len(self.memories) > self.capacity:
            evicted = self._evict()
        return evicted

    def _evict(self) -> list[MemoryItem]:
        """Evict lowest-scored memories to get back to capacity."""
        overflow = len(self.memories) - self.capacity
        to_evict = self.scorer.select_for_eviction(self.memories, evict_count=overflow)
        for m in to_evict:
            self.memories.remove(m)
            self.eviction_log.append({
                "content": m.content,
                "score": self.scorer.score(m),
                "age_days": (datetime.now(timezone.utc) - m.first_seen).days,
                "state": m.state.value,
                "evicted_at": datetime.now(timezone.utc).isoformat(),
            })
        return to_evict

    def retrieve(self, query: str = None, top_k: int = 5) -> list[MemoryItem]:
        """Retrieve top memories by score (simple demo — no embedding search)."""
        ranked = self.scorer.rank(self.memories)
        # Return highest-scored (rank returns ascending)
        return [m for m, _ in reversed(ranked[-top_k:])]

    @property
    def utilization(self) -> float:
        return len(self.memories) / self.capacity


bounded_store = BoundedMemoryStore(capacity=20, scorer=scorer)
print(f"BoundedMemoryStore ready (capacity={bounded_store.capacity})")

## Demo: Noise Injection Experiment

The key finding from the TraceRetain paper: bounded retention only differentiates
from cache heuristics when streams contain **noise** — which real-world memory always does.

Let's inject 75% noise into memory and compare bounded vs unbounded retrieval quality.

In [ ]:
# Generate realistic signal memories (things that should be kept)
SIGNAL_MEMORIES = [
    ("Prefers window seats on flights > 4 hours", "preference", 0.8),
    ("Home airport is SFO", "fact", 0.9),
    ("Always books Marriott when available", "preference", 0.85),
    ("Vegetarian — no meat in meal selections", "fact", 0.95),
    ("Maximum hotel budget: $250/night", "fact", 0.7),
    ("Prefers direct flights, avoids layovers > 2h", "preference", 0.8),
    ("Has Marriott Bonvoy Gold status", "fact", 0.6),
    ("Usually travels on Tuesdays or Wednesdays", "preference", 0.5),
]

# Generate noise memories (things that should be evicted)
NOISE_TEMPLATES = [
    "Mentioned weather was {} in {}",
    "Asked about {} once in passing",
    "Colleague {} recommended {}",
    "Saw an ad for {} airlines",
    "Discussed {} prices briefly",
    "Random observation about {} airport",
]

WEATHER = ["cold", "hot", "rainy", "windy", "sunny"]
CITIES = ["London", "Tokyo", "Mumbai", "Berlin", "Sydney", "Toronto"]
TOPICS = ["rental cars", "train passes", "travel insurance", "lounges", "duty free"]
NAMES = ["Mike", "Alex", "Jordan", "Pat"]
AIRLINES = ["Spirit", "Frontier", "RyanAir", "EasyJet"]

def generate_noise(n: int) -> list[tuple[str, str, float]]:
    """Generate n noise memory tuples (content, category, success_corr)."""
    noise = []
    for _ in range(n):
        template = random.choice(NOISE_TEMPLATES)
        # Fill template based on what it expects
        if "weather" in template:
            content = template.format(random.choice(WEATHER), random.choice(CITIES))
        elif "colleague" in template.lower():
            content = template.format(random.choice(NAMES), random.choice(TOPICS))
        elif "airline" in template:
            content = template.format(random.choice(AIRLINES))
        else:
            content = template.format(random.choice(TOPICS + CITIES))
        noise.append((content, "event", random.uniform(0.0, 0.2)))
    return noise

# Generate 75% noise
n_noise = len(SIGNAL_MEMORIES) * 3  # 75% noise ratio
noise_memories = generate_noise(n_noise)

print(f"Signal memories: {len(SIGNAL_MEMORIES)}")
print(f"Noise memories:  {len(noise_memories)}")
print(f"Noise ratio:     {len(noise_memories) / (len(SIGNAL_MEMORIES) + len(noise_memories)):.0%}")

In [ ]:
def create_memory_item(content: str, category: str, success: float,
                       is_signal: bool, age_range: tuple = (1, 90)) -> MemoryItem:
    """Create a MemoryItem with realistic characteristics."""
    age = random.randint(*age_range)
    return MemoryItem(
        user_id="E001",
        content=content,
        category=category,
        state=MemoryState.TRUSTED if is_signal else MemoryState.CANDIDATE,
        first_seen=now - timedelta(days=age),
        access_count=random.randint(3, 10) if is_signal else random.randint(0, 1),
        last_accessed=(now - timedelta(days=random.randint(1, 10))) if is_signal else None,
        success_correlation=success,
        confirmation_count=random.randint(2, 5) if is_signal else 0,
    )

# Populate bounded store
bounded = BoundedMemoryStore(capacity=20, scorer=scorer)

# Interleave signal and noise (randomised order)
all_items = (
    [(c, cat, s, True) for c, cat, s in SIGNAL_MEMORIES] +
    [(c, cat, s, False) for c, cat, s in noise_memories]
)
random.shuffle(all_items)

for content, category, success, is_signal in all_items:
    m = create_memory_item(content, category, success, is_signal)
    bounded.add(m)

print(f"Store after ingestion:")
print(f"  Stored:  {len(bounded.memories)}")
print(f"  Evicted: {len(bounded.eviction_log)}")
print(f"  Capacity utilization: {bounded.utilization:.0%}")

In [ ]:
# Compare: what's in the bounded store vs what would be in unbounded
signal_in_store = sum(1 for m in bounded.memories if m.state == MemoryState.TRUSTED)
noise_in_store = sum(1 for m in bounded.memories if m.state == MemoryState.CANDIDATE)

print("=== Bounded Store Quality ===")
print(f"  Signal memories retained: {signal_in_store}/{len(SIGNAL_MEMORIES)} "
      f"({signal_in_store/len(SIGNAL_MEMORIES):.0%})")
print(f"  Noise memories retained:  {noise_in_store}/{len(noise_memories)} "
      f"({noise_in_store/len(noise_memories):.0%})")
print(f"  Signal-to-noise ratio:    {signal_in_store}:{noise_in_store}")
print()

print("=== If Unbounded (All Memories Kept) ===")
total = len(SIGNAL_MEMORIES) + len(noise_memories)
print(f"  Signal memories: {len(SIGNAL_MEMORIES)}/{total} ({len(SIGNAL_MEMORIES)/total:.0%})")
print(f"  Noise memories:  {len(noise_memories)}/{total} ({len(noise_memories)/total:.0%})")
print(f"  Signal-to-noise ratio: {len(SIGNAL_MEMORIES)}:{len(noise_memories)}")
print()
print(f"→ Bounded retention concentrates signal from {len(SIGNAL_MEMORIES)/total:.0%} to "
      f"{signal_in_store/max(len(bounded.memories),1):.0%} of stored memories")

In [ ]:
# What got evicted? Show the eviction log
print("=== Eviction Log (last 10) ===")
print()
for entry in bounded.eviction_log[-10:]:
    print(f"  Score: {entry['score']:.3f} | State: {entry['state']:<10} | {entry['content'][:50]}")

print(f"\n→ Notice: evicted items are primarily CANDIDATE state with low scores")

## The Payoff: Retrieval Precision

The real test: when retrieving top-K memories for a query, does bounded retention
return more relevant results than an unbounded store?

In [ ]:
# Retrieve top-5 from the bounded store
top_memories = bounded.retrieve(top_k=5)

print("=== Top 5 Retrieved Memories (Bounded Store) ===")
print()
signal_in_top5 = 0
for i, m in enumerate(top_memories, 1):
    is_signal = m.state == MemoryState.TRUSTED
    signal_in_top5 += int(is_signal)
    icon = "✅" if is_signal else "❌"
    print(f"  {i}. {icon} [{m.state.value:10s}] {m.content[:55]}")
    print(f"       Score: {scorer.score(m):.3f} | Access: {m.access_count}x | "
          f"Success: {m.success_correlation:.1f}")

print(f"\nPrecision@5: {signal_in_top5}/5 = {signal_in_top5/5:.0%}")
print(f"(In unbounded store with 75% noise, expected Precision@5 ≈ 25%)")

## Beyond Eviction: Multi-Level Compression

Instead of just deleting low-value memories, we can **compress** them into summaries.
This preserves information at reduced granularity:

- **Level 0**: Raw individual memories (full detail)
- **Level 1**: Clustered summaries (group related memories)
- **Level 2**: User profile (distilled from all memories)

In [ ]:
COMPRESSION_PROMPT = """You are a memory compression system. Given a list of related memories about a user,
produce a single concise summary that preserves the key information.

Memories to compress:
{memories}

Rules:
- Preserve actionable preferences and constraints
- Drop one-time observations and noise
- Keep specific values (numbers, names, limits)
- Output a single sentence or two maximum

Respond with ONLY the compressed summary text."""


async def compress_memories(memories: list[MemoryItem]) -> str:
    """Compress multiple memories into a single summary."""
    memory_text = "\n".join(f"- {m.content}" for m in memories)
    prompt = COMPRESSION_PROMPT.format(memories=memory_text)
    messages = [
        Message(role="system", contents=[prompt]),
        Message(role="user", contents=["Compress these memories."]),
    ]
    response = await client.get_response(messages=messages)
    return response.text.strip()

print("Compression function ready")

In [ ]:
# Demo: compress related travel preference memories
travel_prefs = [
    MemoryItem(content="Prefers window seats on flights > 4 hours", user_id="E001"),
    MemoryItem(content="Prefers direct flights, avoids layovers > 2h", user_id="E001"),
    MemoryItem(content="Usually travels on Tuesdays or Wednesdays", user_id="E001"),
    MemoryItem(content="Home airport is SFO", user_id="E001"),
]

print("=== Level 0: Raw Memories ===")
for m in travel_prefs:
    print(f"  • {m.content}")

compressed = await compress_memories(travel_prefs)
print(f"\n=== Level 1: Compressed Summary ===")
print(f"  {compressed}")
print(f"\n  (4 memories → 1 summary, ~75% storage reduction)")

In [ ]:
# Full user profile compression (Level 2)
all_signal = [
    MemoryItem(content=c, user_id="E001")
    for c, _, _ in SIGNAL_MEMORIES
]

print("=== Level 2: User Profile Summary ===\n")
print("Source memories:")
for m in all_signal:
    print(f"  • {m.content}")

profile = await compress_memories(all_signal)
print(f"\nDistilled profile:")
print(f"  {profile}")
print(f"\n  ({len(all_signal)} memories → 1 profile summary)")

## Decay Visualization

The exponential decay function determines how quickly recency score drops.
With a 30-day half-life, a memory is worth 50% at 30 days, 25% at 60 days, etc.

In [ ]:
# Visualise the decay curve
half_life = scorer.half_life_days

print(f"Exponential Decay Curve (half-life = {half_life} days)")
print(f"{'Day':<6} {'Score':<8} {'Bar'}")
print("─" * 60)

for day in [0, 5, 10, 15, 30, 45, 60, 90, 120, 180, 365]:
    score = math.exp(-0.693 * day / half_life)
    bar = "█" * int(score * 40)
    print(f"{day:<6} {score:<8.3f} {bar}")

print(f"\n→ At {half_life} days: score = 0.500 (half-life)")
print(f"→ At {half_life*2:.0f} days: score = 0.250")
print(f"→ At {half_life*4:.0f} days: score = 0.063 (likely evicted)")

## Key Takeaways

1. **Bounded > Unbounded** — capacity limits prevent noise accumulation
2. **Multi-dimensional scoring** — not just recency (LRU) or frequency (LFU)
3. **Success correlation is the strongest signal** — memories that led to good outcomes are most valuable
4. **75% noise resilience** — bounded retention maintains precision even under heavy noise
5. **Compress before evict** — multi-level summaries preserve information at reduced granularity
6. **Exponential decay with configurable half-life** — tune to your domain's fact-change rate

## Semantic Memory Lifecycle Complete

Notebooks 01–05 form the complete **semantic memory lifecycle**:

| Notebook | Capability |
|----------|------------|
| 01 | Distinguished memory from RAG and context |
| 02 | Identified what qualifies as memory |
| 03 | Gated trust with staged promotion |
| 04 | Revised beliefs with bi-temporal tracking |
| 05 | Bounded retention with score-based eviction |

Together: **identify → promote → revise → retain/evict**.

## Next: Episodic & Procedural Lifecycles (Notebooks 06–07)

Semantic memory (preferences, facts) has the lifecycle above. But episodic memory
(past events) and procedural memory (learned procedures) need their own lifecycle
strategies — TTL-based expiry and RAG-validated staleness, respectively.